In [11]:
import datetime
import requests
import pymysql
import pandas as pd
from tqdm import tqdm
import time
from DATA.stock_invest_function import get_db_host
from DATA.us_target_ticker_list_2000 import ticker_list

FMP_API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

# 설정
DB_CONFIG = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

# 데이터 수집 기간 설정
START_DATE = '2015-01-01'  # 시작 날짜 (YYYY-MM-DD 형식)

# 테스트 모드 설정
TEST_MODE = True  # True: 테스트 모드 (일부만 수집), False: 전체 수집
TEST_TICKER_COUNT = 30  # 테스트 모드에서 수집할 ticker 개수

def create_database_and_table():
    """테이블 생성 (investar 데이터베이스 사용)"""
    connection = pymysql.connect(**DB_CONFIG)

    try:
        with connection.cursor() as cursor:
            create_table_query = """
            CREATE TABLE IF NOT EXISTS us_stock_market_cap (
                id INT AUTO_INCREMENT PRIMARY KEY,
                ticker VARCHAR(20) NOT NULL,
                date DATE NOT NULL,
                indicator VARCHAR(50) NOT NULL,
                value DOUBLE,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                UNIQUE KEY unique_ticker_date_indicator (ticker, date, indicator),
                INDEX idx_ticker (ticker),
                INDEX idx_date (date)
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
            """
            cursor.execute(create_table_query)
            connection.commit()
            print(f"테이블 생성 완료: {DB_CONFIG['database']}.us_stock_market_cap")
    finally:
        connection.close()

def get_historical_market_cap(ticker, api_key, start_date=START_DATE):
    """FMP API를 통해 특정 ticker의 과거 시가총액 데이터 조회"""
    end_date = datetime.datetime.now()
    start_date_obj = datetime.datetime.strptime(start_date, '%Y-%m-%d')

    # historical-price-full API 사용 (전체 기간 데이터 제공)
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}"
    params = {
        'apikey': api_key,
        'from': start_date_obj.strftime('%Y-%m-%d'),
        'to': end_date.strftime('%Y-%m-%d')
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if 'historical' in data and len(data['historical']) > 0:
            df = pd.DataFrame(data['historical'])
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date')

            # marketCap 컬럼이 있는지 확인
            if 'marketCap' in df.columns:
                df['year_month'] = df['date'].dt.to_period('M')
                monthly_data = df.groupby('year_month').last().reset_index()
                monthly_data['date'] = monthly_data['year_month'].dt.to_timestamp()

                return monthly_data[['date', 'marketCap']].rename(columns={'marketCap': 'value'})
            else:
                print(f"\n{ticker}: marketCap 데이터 없음")
                return None
        else:
            return None

    except requests.exceptions.RequestException as e:
        print(f"\n{ticker} API 요청 실패: {str(e)}")
        return None
    except Exception as e:
        print(f"\n{ticker} 데이터 처리 실패: {str(e)}")
        return None

def insert_market_cap_data(ticker, df_data):
    """시가총액 데이터를 데이터베이스에 저장"""
    if df_data is None or len(df_data) == 0:
        return 0

    connection = pymysql.connect(**DB_CONFIG)

    inserted_count = 0

    try:
        with connection.cursor() as cursor:
            for _, row in df_data.iterrows():
                insert_query = """
                INSERT INTO us_stock_market_cap (ticker, date, indicator, value)
                VALUES (%s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE value = VALUES(value)
                """
                cursor.execute(insert_query, (
                    ticker,
                    row['date'].strftime('%Y-%m-%d'),
                    'market_cap',
                    float(row['value']) if pd.notna(row['value']) else None
                ))
                inserted_count += 1

            connection.commit()
    except Exception as e:
        print(f"\n{ticker} DB 저장 실패: {str(e)}")
        connection.rollback()
        inserted_count = 0
    finally:
        connection.close()

    return inserted_count

def main():
    # 테스트 모드에 따라 ticker 리스트 결정
    if TEST_MODE:
        tickers_to_process = ticker_list[:TEST_TICKER_COUNT]
        mode_text = f"테스트 모드 - {TEST_TICKER_COUNT}개"
    else:
        tickers_to_process = ticker_list
        mode_text = "전체 수집"

    print("=" * 80)
    print("미국 주식 시가총액 데이터 수집 시작")
    print("=" * 80)
    print(f"모드: {mode_text}")
    print(f"총 Ticker 수: {len(tickers_to_process)}")
    print(f"수집 기간: {START_DATE} ~ 현재 (월별 데이터)")
    print(f"저장 위치: {DB_CONFIG['database']}.us_stock_market_cap")
    print("=" * 80)

    create_database_and_table()

    total_inserted = 0
    success_count = 0
    fail_count = 0

    print("\n데이터 수집 시작...\n")

    for ticker in tqdm(tickers_to_process, desc="진행 상황", unit="ticker"):
        df_market_cap = get_historical_market_cap(ticker, FMP_API_KEY)

        if df_market_cap is not None and len(df_market_cap) > 0:
            inserted = insert_market_cap_data(ticker, df_market_cap)
            total_inserted += inserted
            success_count += 1
            time.sleep(0.2)
        else:
            fail_count += 1
            time.sleep(0.2)

    print("\n" + "=" * 80)
    print("데이터 수집 완료")
    print("=" * 80)
    print(f"성공: {success_count} ticker")
    print(f"실패: {fail_count} ticker")
    print(f"총 저장 레코드 수: {total_inserted:,}")
    print("=" * 80)

    connection = pymysql.connect(**DB_CONFIG)

    try:
        with connection.cursor() as cursor:
            cursor.execute("SELECT COUNT(*) FROM us_stock_market_cap")
            total_records = cursor.fetchone()[0]

            cursor.execute("SELECT COUNT(DISTINCT ticker) FROM us_stock_market_cap")
            unique_tickers = cursor.fetchone()[0]

            cursor.execute("SELECT MIN(date), MAX(date) FROM us_stock_market_cap")
            date_range = cursor.fetchone()

            print(f"\n[데이터베이스 확인]")
            print(f"총 레코드 수: {total_records:,}")
            print(f"고유 Ticker 수: {unique_tickers}")
            print(f"데이터 기간: {date_range[0]} ~ {date_range[1]}")

            cursor.execute("""
                SELECT ticker, COUNT(*) as cnt
                FROM us_stock_market_cap
                GROUP BY ticker
                ORDER BY cnt DESC
                LIMIT 5
            """)
            print(f"\n[샘플 Ticker 데이터 건수]")
            for row in cursor.fetchall():
                print(f"  {row[0]}: {row[1]} 건")
    finally:
        connection.close()

if __name__ == "__main__":
    main()

미국 주식 시가총액 데이터 수집 시작
모드: 테스트 모드 - 30개
총 Ticker 수: 30
수집 기간: 2015-01-01 ~ 현재 (월별 데이터)
저장 위치: investar.us_stock_market_cap
테이블 생성 완료: investar.us_stock_market_cap

데이터 수집 시작...



진행 상황:   0%|          | 0/30 [00:00<?, ?ticker/s]


NVDA: marketCap 데이터 없음


진행 상황:   3%|▎         | 1/30 [00:01<00:45,  1.59s/ticker]


GOOG: marketCap 데이터 없음


...


CAT: marketCap 데이터 없음


진행 상황:  97%|█████████▋| 29/30 [00:45<00:01,  1.56s/ticker]


TM: marketCap 데이터 없음


진행 상황: 100%|██████████| 30/30 [00:47<00:00,  1.57s/ticker]


데이터 수집 완료
성공: 0 ticker
실패: 30 ticker
총 저장 레코드 수: 0

[데이터베이스 확인]
총 레코드 수: 1,260
고유 Ticker 수: 20
데이터 기간: 2020-11-01 ~ 2026-01-01

[샘플 Ticker 데이터 건수]
  ASML: 63 건
  MSFT: 63 건
  XOM: 63 건
  COST: 63 건
  NVDA: 63 건
